# Notebook 02 : les outils de l'agent, testés sans modèle de langage

**Projet 6, groupe 6.** Les trois outils `@tool` du projet (`tournoi_axelrod`, `replicateur`, `seuils_actualisation`) sont appelés ici avec `.invoke()`, sur des entrées valides puis invalides. La fin du notebook fait tourner la boucle de l'agent avec un **faux modèle étiqueté « scripté »** : cela prouve que la mécanique fonctionne (outil appelé, résultat renvoyé, réponse produite), **jamais** la qualité d'un vrai LLM. Aucune sortie de ce notebook ne vient d'Ollama.

In [1]:
import sys, json
sys.path.insert(0, "../src")
from agent_tj_sma import (OUTILS, tournoi_axelrod, replicateur_outil, seuils_actualisation, construire_agent, interroger,
                          PLAFOND_MANCHES, PLAFOND_GENERATIONS, PLAFOND_REPETITIONS, SYSTEM_PROMPT)
from modele_scripte import ModeleScripte, appel_outil, reponse_finale
print("plafonds : manches", PLAFOND_MANCHES, "| générations", PLAFOND_GENERATIONS, "| répétitions", PLAFOND_REPETITIONS)

plafonds : manches 1000 | générations 2000 | répétitions 20


## 1. Ce que le modèle voit de chaque outil

Le modèle ne lit ni le code ni les tests : il lit le **nom**, la **docstring** et les **types** des arguments. Une docstring vague est la première cause d'un agent qui n'appelle aucun outil.

In [2]:
for o in OUTILS:
    print(f"- {o.name}  arguments : { {k: v.get('type', v.get('anyOf')) for k, v in o.args.items()} }")
    print("   ", o.description.splitlines()[0])

- tournoi_axelrod  arguments : {'strategies': 'array', 'manches': 'integer', 'bruit': 'number', 'seed': 'integer', 'repetitions': 'integer'}
    Simule un tournoi d'Axelrod du dilemme du prisonnier répété entre des stratégies connues.
- replicateur  arguments : {'strategies': 'array', 'generations': 'integer', 'manches': 'integer', 'bruit': 'number', 'seed': 'integer', 'repetitions': 'integer'}
    Fait évoluer une population de stratégies par la dynamique du réplicateur, à partir de parts égales.
- seuils_actualisation  arguments : {'t': 'number', 'r': 'number', 'p': 'number', 's': 'number'}
    Calcule les seuils d'actualisation delta au-delà desquels la coopération est stable à horizon indéterminé.


## 2. Appels avec `.invoke()` (entrées valides)

In [3]:
r = json.loads(tournoi_axelrod.invoke({"strategies": ["TFT", "toujours_D", "Pavlov"], "manches": 200, "bruit": 0.0, "seed": 0}))
print("paramètres :", r["parametres"])
for ligne, n in zip(r["matrice"], r["noms"]):
    print(f"{n:>10}", [round(v, 3) for v in ligne])
print("classement :", [(c["strategie"], round(c["score_moyen"], 3)) for c in r["classement"]])

paramètres : {'manches': 200, 'bruit': 0.0, 'seed': 0, 'repetitions': 1}
       TFT [3.0, 0.995, 3.0]
toujours_D [1.02, 1.0, 3.0]
    Pavlov [3.0, 0.5, 3.0]
classement : [('TFT', 2.332), ('Pavlov', 2.167), ('toujours_D', 1.673)]


In [4]:
r = json.loads(replicateur_outil.invoke({"strategies": [], "generations": 200, "manches": 200, "bruit": 0.01, "seed": 0, "repetitions": 5}))
print("parts finales :", {k: round(v, 3) for k, v in r["parts_finales"].items()}, "| somme :", round(r["somme_parts_finales"], 12))
print("points de la trajectoire :", list(r["trajectoire"]))

parts finales : {'toujours_C': 0.0, 'toujours_D': 0.0, 'TFT': 0.0, 'rancunier': 0.563, 'Pavlov': 0.437, 'aleatoire': 0.0} | somme : 1.0
points de la trajectoire : ['0', '20', '40', '60', '80', '100', '120', '140', '160', '180', '200']


In [5]:
print(json.loads(seuils_actualisation.invoke({})))
print(json.loads(seuils_actualisation.invoke({"t": 5, "r": 2, "p": 1, "s": 0})))

{'outil': 'seuils_actualisation', 'dilemme': True, 'gains': {'T': 5.0, 'R': 3.0, 'P': 1.0, 'S': 0.0}, 'seuil_rancunier': 0.5, 'seuil_tft': 0.6666666666666666}
{'outil': 'seuils_actualisation', 'dilemme': False, 'message': "Les gains ne vérifient pas T > R > P > S et 2R > T + S : ce n'est pas un dilemme répété."}


## 3. Entrées invalides et plafonds

Deux niveaux de protection : le **schéma** (types annotés) refuse un mauvais type avant l'exécution ; l'outil refuse ensuite une valeur invalide en renvoyant `{"erreur": ...}`, que l'agent peut lire et expliquer. Les paramètres coûteux sont **plafonnés** et l'ajustement est signalé.

In [6]:
print(json.loads(tournoi_axelrod.invoke({"strategies": ["__import__('os').system('echo pirate')"]})))   # jamais exécuté
print(json.loads(tournoi_axelrod.invoke({"strategies": ["TFT"], "bruit": 1.5})))
try:
    tournoi_axelrod.invoke({"strategies": "TFT"})
except Exception as e:
    print("schéma :", type(e).__name__)
r = json.loads(tournoi_axelrod.invoke({"strategies": ["TFT", "Pavlov"], "manches": 10**9, "repetitions": 10**6}))
print(r["parametres"], r["avertissements"])

{'erreur': 'stratégie(s) inconnue(s) : ["__import__(\'os\').system(\'echo pirate\')"] ; connues : [\'Pavlov\', \'TFT\', \'TFT_genereux\', \'aleatoire\', \'rancunier\', \'toujours_C\', \'toujours_D\']'}
{'erreur': 'bruit doit être dans [0, 1]'}
schéma : ValidationError


{'manches': 1000, 'bruit': 0.0, 'seed': 0, 'repetitions': 20} ['manches ramené de 1000000000 à 1000 (plafond 1000)', 'repetitions ramené de 1000000 à 20 (plafond 20)']


## 4. La boucle de l'agent avec le faux modèle (scripté)

Le script prépare à l'avance deux appels d'outils puis une réponse. Chaque sortie du faux modèle commence par `[scripté]`.

In [7]:
agent = construire_agent(ModeleScripte(reponses=[
    appel_outil("seuils_actualisation", {}, "a1"),
    appel_outil("tournoi_axelrod", {"strategies": ["TFT", "toujours_D"], "manches": 40, "bruit": 0.0, "seed": 0}, "a2"),
    reponse_finale("Seuils et tournoi calculés par les outils ; voir les résultats ci-dessus."),
]))
messages = interroger(agent, "Donne les seuils d'actualisation puis compare TFT et toujours_D sur 40 manches.", verbose=True)
print("\nnombre de messages échangés :", len(messages), "| outils appelés :", [m.name for m in messages if m.type == "tool"])

[appel d'outil] seuils_actualisation({})
[réponse] [scripté] appel de l'outil seuils_actualisation
[résultat de seuils_actualisation] {"outil": "seuils_actualisation", "dilemme": true, "gains": {"T": 5.0, "R": 3.0, "P": 1.0, "S": 0.0}, "seuil_rancunier": 0.5, "seuil_tft": 0.6666666666666666}
[appel d'outil] tournoi_axelrod({'strategies': ['TFT', 'toujours_D'], 'manches': 40, 'bruit': 0.0, 'seed': 0})
[réponse] [scripté] appel de l'outil tournoi_axelrod
[résultat de tournoi_axelrod] {"outil": "tournoi_axelrod", "parametres": {"manches": 40, "bruit": 0.0, "seed": 0, "repetitions": 1}, "noms": ["TFT", "toujours_D"], "matrice": [[3.0, 0.975], [1.1, 1.0]], "classement": [{"strategie": "TFT", "score_moyen": 1.9875}, {"strategie": "toujours_D", "score_moyen": 1.05}], "avertissements"
[réponse] [scripté] Seuils et tournoi calculés par les outils ; voir les résultats ci-dessus.

nombre de messages échangés : 6 | outils appelés : ['seuils_actualisation', 'tournoi_axelrod']


## 5. Ce que ces tests prouvent, et ce qu'ils ne prouvent pas

Ils prouvent que les outils valident leurs entrées, plafonnent les paramètres coûteux, renvoient du JSON lisible et que la boucle de l'agent exécute un appel d'outil puis répond. Ils ne disent **rien** sur la capacité d'un vrai modèle à choisir le bon outil, à écrire les bons arguments ou à lire correctement le résultat : cela se mesure avec Ollama (`verbose=True`), avec plusieurs essais par question, dans une étape ultérieure du projet.